# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [3]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

# Entidad esperada para la validación de logos.
ENTIDAD_ESPERADA = "PROINVERSIÓN"

# Para prueba rápida puedes poner "10". Para procesar todo el PDF usa "None".
MAX_PAGES_PAGINAS = "10"


## 2. (Opcional) Verificar GPU

In [4]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

Sat Jul 11 22:42:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Clonar el repositorio

In [5]:
%cd /content
!rm -rf /content/ramec
!git clone https://github.com/{REPO}.git /content/ramec
%cd /content/ramec
!grep -n "Firma_Elaboró" src/report.py
!grep -n "analizar_firmas_control" src/extract.py
!grep -n "VAL_CTRL" src/infer.py

/content
Cloning into '/content/ramec'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 127 (delta 62), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 117.76 KiB | 5.61 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/ramec
175:        "Firma_Elaboró": _sn(firmas[0]),
707:            "Firma_Elaboró", "Firma_Revisó", "Firma_Aprobó_1", "Firma_Aprobó_2",
674:def analizar_firmas_control(crop, filas_esperadas=None):
36:VAL_CTRL = C.NAME_TO_ID["validacion_profesional_hoja_control"]  # 9
177:        if VAL_CTRL in boxes:
178:            firma_crop = EX.crop_box(img, boxes[VAL_CTRL][0])


## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [6]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-spa.
Preparing to unpa

## 5. Dependencias de Python

In [7]:
!pip -q install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.0 MB/s eta 0:00:00


## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [8]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

Descargando pesos desde https://github.com/BRIDERI/Ramec/releases/download/v1.0 ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.8M  100 38.8M    0     0  28.8M      0  0:00:01  0:00:01 --:--:-- 52.4M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.7M  100 38.7M    0     0  14.7M      0  0:00:02  0:00:02 --:--:-- 18.8M
Listo. Pesos descargados:
-rw-r--r-- 1 root root 39M Jul 11 22:43 models/documentos/best.pt
-rw-r--r-- 1 root root 39M Jul 11 22:43 models/planos/best.pt


## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [9]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

Selecciona uno o más PDFs...


Saving AVP-SAV-1000-D-INF-DSA-ING-000002.pdf to AVP-SAV-1000-D-INF-DSA-ING-000002.pdf
total 37M
-rw-r--r-- 1 root root 37M Jul 11 22:45 AVP-SAV-1000-D-INF-DSA-ING-000002.pdf


*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [10]:
!python src/infer.py \
  --pdfs pdfs \
  --salida outputs/Reporte_validacion.xlsx \
  --entidad-esperada "{ENTIDAD_ESPERADA}" \
  --max-pages-paginas {MAX_PAGES_PAGINAS}


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  validación complementaria por página: AVP-SAV-1000-D-INF-DSA-ING-000002.pdf
Listo: outputs/Reporte_validacion.xlsx
  planos=0 documentos=1 validacion_profesional=1 filas estandar=1 paginas=10


## 9. Ver y descargar el reporte

Se genera un Excel con hojas resumen y hojas técnicas ocultas para auditoría.


In [11]:
%cd /content/ramec

import pandas as pd
from google.colab import files

REPORTE = "outputs/Reporte_validacion.xlsx"

xls = pd.ExcelFile(REPORTE)
print("Hojas:", xls.sheet_names)

if "GENERAL" in xls.sheet_names:
    print("\nGENERAL:")
    display(pd.read_excel(xls, "GENERAL"))

if "VALIDACION_PAGINAS" in xls.sheet_names:
    print("\nVALIDACION_PAGINAS:")
    display(pd.read_excel(xls, "VALIDACION_PAGINAS"))

if "VALIDACION_LOGOS" in xls.sheet_names:
    print("\nVALIDACION_LOGOS:")
    display(pd.read_excel(xls, "VALIDACION_LOGOS"))

files.download(REPORTE)


/content/ramec
Hojas: ['GENERAL', 'ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL', 'VALIDACION_PAGINAS', 'VALIDACION_LOGOS', 'VALIDACION_FINAL_PAGINAS', 'DETALLE_SELLOS_PAGINAS', 'DETALLE_LOGOS_PAGINAS']

GENERAL:


,Ruta,Archivo,ESTANDAR NOMENCLATURA,COMPATIBILIDAD_DOCUMENTO,COHERENCIA_DOCUMENTO,CONTROL_CAMBIOS_DOC,VALIDACION_PROFESIONAL,VALIDACION_PAGINAS,VALIDACION_LOGOS
0,pdfs/AVP-SAV-1000-D-INF-DSA-ING-000002.pdf,AVP-SAV-1000-D-INF-DSA-ING-000002.pdf,OK,OK,NO,OK,NO,OK,NO



VALIDACION_PAGINAS:


,Archivo,Sello_detectado_en_contenido,%_páginas_con_sello,Cantidad_páginas_con_sello,Firmantes_detectados,Cargos_detectados,Responsables_hoja_de_control,Responsables_detectados,Responsables_no_detectados,Firmantes_coinciden,CIP_detectados,CUMPLE,Observación_general
0,AVP-SAV-1000-D-INF-DSA-ING-000002.pdf,Sí,100,8,Alfredo Falconi Guerrero | Miguel Núñez Gutiér...,Gerente de Construcción | Gerente de Proyecto ...,Alfredo Falconi Guerrero | Germán Asenjo Quisp...,Alfredo Falconi Guerrero | Germán Asenjo Quisp...,NaN,OK,264301 | 199335,OK,Se detectaron sellos laterales en todas las pá...



VALIDACION_LOGOS:


,Archivo,Logo_detectado_documento,%_páginas_con_logo,Entidad_esperada,Entidades_detectadas,Concedente_OK,CUMPLE,Observación_general
0,AVP-SAV-1000-D-INF-DSA-ING-000002.pdf,Sí,100,PROINVERSIÓN,MTC | Sociedad Concesionaria Anillo Vial,NO,NO,Se detectó logo en todas las páginas evaluadas...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [ ]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt